# 04c — TGN-no-mem: Per-Instance Graph

**Ablation baseline — same graph as `04_tgn_training.ipynb` (TGN-id).**

## Model: TGN-no-mem (Rossi et al. 2020, Table 2)
```
No GRU memory. No graph operator.
embed = MLP( [sensor_norm, delta_t] )
pred  = sigmoid( classifier(embed) )
```
Pure edge-feature classifier: if TGN-id/TGN-time outperform this,
the GRU memory is contributing to anomaly detection.

## Graph: per-instance bipartite (same as 04_tgn_training)

Compare with: `04_tgn_training.ipynb` (TGN-id) | `04b_tgn_time.ipynb` (TGN-time)

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
import sys
sys.path.append('../../src')
from config import DATA_ROOT_3W, PROCESSED_DIR

torch.manual_seed(42)
np.random.seed(42)

PROCESSED_DIR = Path('../../data/processed')
MODELS_DIR    = Path('../../models')
MODELS_DIR.mkdir(exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


PyTorch: 2.9.1+cu128
Device: GPU


## 1. Load Data from Neo4j

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED_DIR = Path('/home/obiaggi/3w_processed')
MODELS_DIR    = Path('/home/obiaggi/TKG_Thesis/models')
MODELS_DIR.mkdir(exist_ok=True, parents=True)

SENSOR_COLS = ['P-PDG', 'P-TPT', 'T-TPT', 'P-MON-CKP', 'T-JUS-CKP', 'P-JUS-CKGL', 'QGL', 'T-PDG']
ROWS_PER_CLASS = 2000

CLASS_NAMES = {
    0:'Normal', 1:'BSW increase', 2:'DHSV closure', 3:'Severe slugging',
    4:'Flow instability', 5:'Productivity loss', 6:'Quick restriction',
    7:'Scaling', 8:'Hydrate', 9:'Other'
}

dfs = []
for type_num in range(10):
    f = PROCESSED_DIR / f'type_{type_num}.parquet'
    if not f.exists():
        print(f'  Missing type_{type_num} — skipping')
        continue
    d = pd.read_parquet(f)
    if 'event_type' in d.columns and 'class' not in d.columns:
        d = d.rename(columns={'event_type': 'class'})
    # Preserve per-well instance identity — critical for proper TGN memory isolation
    if 'instance_id' not in d.columns:
        d['instance_id'] = f'WELL-{type_num:02d}'
    else:
        d['instance_id'] = d['instance_id'].astype(str)
    available = [c for c in SENSOR_COLS if c in d.columns]
    d = d.dropna(subset=available, how='all')
    if len(d) > ROWS_PER_CLASS:
        d = d.sample(ROWS_PER_CLASS, random_state=42)
    d['is_anomaly'] = (d['class'].fillna(0) > 0).astype(int)
    d['type_num'] = type_num
    dfs.append(d)
    n_inst = d['instance_id'].nunique()
    print(f'  type_{type_num} ({CLASS_NAMES[type_num]:<20}): {len(d):>6,} rows | {n_inst:>3} instances | anomaly: {d["is_anomaly"].mean()*100:.0f}%')

df = pd.concat(dfs, ignore_index=True)
available_sensors = [c for c in SENSOR_COLS if c in df.columns]

print(f'\nTotal rows  : {len(df):,}')
print(f'Normal      : {(df["is_anomaly"]==0).sum():,}')
print(f'Anomaly     : {(df["is_anomaly"]==1).sum():,}  ({df["is_anomaly"].mean()*100:.1f}%)')
print(f'Well instances: {df["instance_id"].nunique()} | Sensor types: {len(available_sensors)}')

  type_0 (Normal              ):  2,000 rows | 578 instances | anomaly: 0%


  type_1 (BSW increase        ):  2,000 rows | 128 instances | anomaly: 89%


  type_2 (DHSV closure        ):  2,000 rows |  38 instances | anomaly: 70%


  type_3 (Severe slugging     ):  2,000 rows | 106 instances | anomaly: 97%


  type_4 (Flow instability    ):  2,000 rows | 341 instances | anomaly: 67%


  type_5 (Productivity loss   ):  2,000 rows | 439 instances | anomaly: 97%


  type_6 (Quick restriction   ):  2,000 rows | 221 instances | anomaly: 92%


  type_7 (Scaling             ):  2,000 rows |  46 instances | anomaly: 85%


  type_8 (Hydrate             ):  2,000 rows |  95 instances | anomaly: 85%


  type_9 (Other               ):  2,000 rows | 205 instances | anomaly: 64%

Total rows  : 20,000
Normal      : 5,073
Anomaly     : 14,927  (74.6%)
Well instances: 1541 | Sensor types: 8


## 2. Prepare TGN Data — Per-Instance Bipartite Graph

**Graph structure (corrected):**

```
well_inst_001 → P-PDG     (edge_feat: value, delta_t)
well_inst_001 → P-TPT
...
well_inst_001 → T-PDG
well_inst_002 → P-PDG
...
```

Each well instance node accumulates its **own** GRU memory, isolated from other instances.
Self-attention on the 8 sensor memories for a given instance captures cross-sensor correlations in time.

Split is at **instance level** (not row level) to prevent temporal leakage.

In [3]:
from sklearn.model_selection import train_test_split as sk_split
from sklearn.preprocessing import StandardScaler
import torch


def prepare_tgn_data(df: pd.DataFrame):
    """
    Bipartite graph: well_instance → sensor_name
    
    Each row (timestep) produces len(sensor_cols) edges, one per sensor.
    Memory for each well_instance accumulates only that instance's history,
    enabling TGN to learn anomaly patterns from temporal state evolution.
    """
    sensor_cols = [c for c in SENSOR_COLS if c in df.columns]

    # Melt: each timestep row → N sensor edges
    id_vars = ['instance_id', 'is_anomaly']
    melted = (
        df[id_vars + sensor_cols]
        .melt(id_vars=id_vars, value_vars=sensor_cols,
              var_name='sensor_name', value_name='sensor_value')
        .dropna(subset=['sensor_value'])
        .reset_index(drop=True)
    )

    # Per-instance normalized timestep: delta_t ∈ [0, 1]
    melted['row_rank'] = melted.groupby('instance_id').cumcount()
    melted['inst_len'] = melted.groupby('instance_id')['row_rank'].transform('max') + 1
    melted['delta_t']  = melted['row_rank'] / melted['inst_len']

    # Normalize sensor values globally
    scaler = StandardScaler()
    melted['sensor_norm'] = scaler.fit_transform(melted[['sensor_value']])

    # Node index: well instances first, then sensor types
    instances = melted['instance_id'].unique().tolist()
    sensors   = sensor_cols
    node2idx  = {n: i for i, n in enumerate(instances + sensors)}
    num_nodes = len(instances) + len(sensors)

    # Stratified split at INSTANCE level — no instance spans both splits
    inst_labels = (
        df.groupby('instance_id')['is_anomaly']
        .max()
        .reset_index(name='label')
    )
    train_insts, test_insts = sk_split(
        inst_labels['instance_id'].values,
        test_size=0.3, random_state=42,
        stratify=inst_labels['label'].values
    )
    train_set = set(train_insts)
    test_set  = set(test_insts)

    train_df = melted[melted['instance_id'].isin(train_set)]
    test_df  = melted[melted['instance_id'].isin(test_set)]

    def to_tensors(d):
        src = torch.tensor([node2idx[w] for w in d['instance_id']], dtype=torch.long)
        dst = torch.tensor([node2idx[s] for s in d['sensor_name']], dtype=torch.long)
        ef  = torch.tensor(d[['sensor_norm', 'delta_t']].values, dtype=torch.float32)
        dt  = torch.tensor(d['delta_t'].values, dtype=torch.float32).unsqueeze(1)
        y   = torch.tensor(d['is_anomaly'].values.astype(float), dtype=torch.float32)
        return src, dst, ef, dt, y

    print(f'Graph: {num_nodes} nodes ({len(instances)} well instances + {len(sensors)} sensor types)')
    print(f'Train edges: {len(train_df):,} | Test edges: {len(test_df):,}')
    print(f'Train anomaly: {train_df["is_anomaly"].mean()*100:.1f}% | '
          f'Test anomaly: {test_df["is_anomaly"].mean()*100:.1f}%')

    return to_tensors(train_df), to_tensors(test_df), num_nodes


train_data, test_data, num_nodes = prepare_tgn_data(df)

Graph: 1549 nodes (1541 well instances + 8 sensor types)
Train edges: 68,933 | Test edges: 36,362
Train anomaly: 73.2% | Test anomaly: 76.7%


## 3. Train TGN-no-mem (edge features only — ablation baseline)

In [4]:
import torch, sys, numpy as np
from sklearn.metrics import classification_report, roc_auc_score, f1_score
sys.path.insert(0, '../../src')
try:
    from models.tgn import TGNNoMem
    print('TGNNoMem imported from src/models')
except ImportError as e:
    print(f'Import error: {e}')

if not df.empty and train_data is not None:
    model = TGNNoMem(num_nodes=num_nodes, memory_dim=64, message_dim=64, embed_dim=64, edge_dim=2)
    print(f'TGN-no-mem parameters: {sum(p.numel() for p in model.parameters()):,}')
    print(f'Graph nodes          : {num_nodes} (unused — no memory)')

    src, dst, ef, dt, y_tr = train_data
    n_normal  = float((y_tr == 0).sum())
    n_anomaly = float((y_tr == 1).sum())
    normal_weight = n_anomaly / max(n_normal, 1)
    print(f'Class weights — Normal: {normal_weight:.2f}x | Anomaly: 1.00x')

    import torch.nn as nn
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss(reduction='none')

    print('\nTraining TGN-no-mem (5 epochs) — edge features only...')
    for epoch in range(5):
        model.train()
        total_loss, n_batch = 0.0, 0
        for i in range(0, len(src), 512):
            s, d = src[i:i+512], dst[i:i+512]
            e, t_ = ef[i:i+512], dt[i:i+512]
            yb = y_tr[i:i+512]
            optimizer.zero_grad()
            pred = model(s, d, e, t_)   # no embed_src needed (no memory)
            w = torch.where(yb == 0,
                            torch.full_like(yb, normal_weight),
                            torch.ones_like(yb))
            loss = (criterion(pred, yb) * w).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            n_batch += 1
        print(f'  Epoch {epoch+1}/5 — loss: {total_loss/n_batch:.4f}')

    model.eval()
    src_t, dst_t, ef_t, dt_t, y_te = test_data
    all_scores = []
    with torch.no_grad():
        for i in range(0, len(src_t), 512):
            s = model(src_t[i:i+512], dst_t[i:i+512],
                      ef_t[i:i+512], dt_t[i:i+512])
            all_scores.extend(s.numpy())

    scores = np.array(all_scores)
    y_np   = y_te.numpy()

    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.1, 0.9, 0.02):
        s = f1_score(y_np, (scores > t).astype(int), average='macro', zero_division=0)
        if s > best_f1:
            best_f1, best_t = s, t

    y_pred = (scores > best_t).astype(int)
    try:
        auc = roc_auc_score(y_np, scores)
    except Exception:
        auc = float('nan')

    print(f'\n[PER-INSTANCE GRAPH — TGN-no-mem]')
    print(f'Best threshold: {best_t:.2f} (macro F1={best_f1:.3f})')
    print(classification_report(y_np, y_pred, target_names=['Normal', 'Anomaly'], digits=3))
    print(f'AUC-ROC: {auc:.4f}')
    print('Note: if AUC ~ TGN-id/time → memory adds no value on this dataset')

    torch.save({'model_state': model.state_dict(), 'num_nodes': num_nodes,
                'dataset': '3W', 'graph': 'per-instance', 'variant': 'TGN-no-mem'},
               MODELS_DIR / 'tgn_3w_nomem.pt')
    print('Checkpoint saved.')
else:
    print('Load data first')

TGNNoMem imported from src/models
TGN-no-mem parameters: 6,529
Graph nodes          : 1549 (unused — no memory)
Class weights — Normal: 2.74x | Anomaly: 1.00x



Training TGN-no-mem (5 epochs) — edge features only...


  Epoch 1/5 — loss: 1.0388


  Epoch 2/5 — loss: 1.0246


  Epoch 3/5 — loss: 1.0309


  Epoch 4/5 — loss: 1.0248


  Epoch 5/5 — loss: 1.0237



[PER-INSTANCE GRAPH — TGN-no-mem]
Best threshold: 0.10 (macro F1=0.434)
              precision    recall  f1-score   support

      Normal      0.000     0.000     0.000      8486
     Anomaly      0.767     1.000     0.868     27876

    accuracy                          0.767     36362
   macro avg      0.383     0.500     0.434     36362
weighted avg      0.588     0.767     0.665     36362

AUC-ROC: 0.4974
Note: if AUC ~ TGN-id/time → memory adds no value on this dataset
Checkpoint saved.


/home/obiaggi/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/obiaggi/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/obiaggi/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
